In [3]:
from wandb_utils import find_best_run, calculate_mean_values, calculate_group_scores

In [13]:
# group_names_old_split = [
#     "last_bc",
#     "last_dqn",
#     "last_cql",
#     "last_sac",
#     "last_min",
#     "last_var",
#     "last_rem",
#     "first_bootstrapping_0.5",
#     "random_runs_diversification_seeds_90_0.2",
#     "random_runs_random_priors_seeds_90_0.2",
# ]

group_names_baselines = [
    "random_runs_bc_seeds",
    "random_runs_dqn_seeds",
    "random_runs_cql_seeds",
    "random_runs_dsac_seeds",
]

group_names_dqn = [
    "random_runs_dqn_seeds",
    "random_runs_min_seeds",
    "random_runs_var_seeds",
    "random_runs_bootstrapping_seeds",
    "random_runs_div_seeds",
    "random_runs_priors_seeds",
    "random_runs_weak_learners_32_hyp"
]

group_names_dsac = [
    "random_runs_dsac_seeds",
    "random_runs_dsac_seeds_min",
    "random_runs_dsac_seeds_var",
    "random_runs_dsac_seeds_bootstrapping",
    "random_runs_dsac_seeds_div",
    "random_runs_dsac_seeds_prior",
    "random_runs_dsac_seeds_small"
]

In [9]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set a modern style
sns.set_theme(style="whitegrid")

def plot_seq_rewards_vs_episodes(group_names, baseline_idx=0, n_finish=None, smooth_window=5):
    group_rewards = {}

    # Identify the baseline group
    baseline_group = group_names[baseline_idx]
    print(f"Baseline group: {baseline_group}")

    for group_name in group_names:
        print(f"Processing group: {group_name}")

        # Filter runs by group name
        filtered_runs = [run for run in runs if run.group == group_name]

        # Store rewards per episode across runs
        all_seq_rewards = []

        for run in filtered_runs:
            history = run.history()
            seq_rewards = history["seq_reward"].values  # Extract seq_reward values
            
            if n_finish is not None:
                seq_rewards = seq_rewards[:n_finish]  # Trim if needed

            all_seq_rewards.append(seq_rewards)

        # Find max length among all runs
        max_length = max(len(seq) for seq in all_seq_rewards)

        # Pad all sequences to the same length with NaNs
        all_seq_rewards_padded = np.full((len(all_seq_rewards), max_length), np.nan)
        for i, seq in enumerate(all_seq_rewards):
            all_seq_rewards_padded[i, :len(seq)] = seq

        # Compute mean and std, ignoring NaNs
        mean_seq_rewards = np.nanmean(all_seq_rewards_padded, axis=0)
        std_seq_rewards = np.nanstd(all_seq_rewards_padded, axis=0)

        # Apply smoothing (rolling average)
        if smooth_window > 1:
            mean_seq_rewards = pd.Series(mean_seq_rewards).rolling(smooth_window, min_periods=1).mean().values
            std_seq_rewards = pd.Series(std_seq_rewards).rolling(smooth_window, min_periods=1).mean().values

        # Store results
        group_rewards[group_name] = (mean_seq_rewards, std_seq_rewards)

    # Set up subplots
    num_groups = len(group_names) - 1  # Exclude the baseline from its own plot
    fig, axes = plt.subplots(nrows=num_groups, ncols=1, figsize=(10, 4 * num_groups), sharex=True)

    if num_groups == 1:
        axes = [axes]  # Ensure `axes` is iterable for a single plot

    episodes = np.arange(1, len(next(iter(group_rewards.values()))[0]) + 1)

    colors = sns.color_palette("husl", num_groups + 1)  # Color palette for groups

    for ax, (group_name, color) in zip(axes, zip(group_names, colors)):
        if group_name == baseline_group:
            continue  # Skip baseline in its own subplot

        mean_rewards, std_rewards = group_rewards[group_name]
        mean_baseline, std_baseline = group_rewards[baseline_group]

        # Plot main group
        ax.plot(episodes, mean_rewards, label=group_name, color=color, linewidth=2)
        ax.fill_between(episodes, mean_rewards - std_rewards, mean_rewards + std_rewards, color=color, alpha=0.2)

        # Always plot baseline for comparison
        ax.plot(episodes, mean_baseline, label=f"Baseline: {baseline_group}", color="black", linestyle="dashed", linewidth=2)
        ax.fill_between(episodes, mean_baseline - std_baseline, mean_baseline + std_baseline, color="gray", alpha=0.2)

        ax.set_ylabel("Mean Seq Reward", fontsize=12)
        ax.set_title(f"{group_name} vs {baseline_group}", fontsize=14, fontweight="bold")
        ax.legend(fontsize=11)
        ax.grid(True, linestyle="--", linewidth=0.6, alpha=0.7)

    plt.xlabel("Episodes", fontsize=12)
    plt.suptitle("Episode vs. Reward for Different Groups", fontsize=16, fontweight="bold")
    plt.tight_layout()
    plt.show()


In [14]:
calculate_mean_values(group_names_dsac, n_included_runs=10, n_finish=101, baseline_idx=0, originality=True)

random_runs_dsac_seeds
random_runs_dsac_seeds_min
random_runs_dsac_seeds_var
random_runs_dsac_seeds_bootstrapping
random_runs_dsac_seeds_div
random_runs_dsac_seeds_prior
random_runs_dsac_seeds_small


,Group Name,Mean Seq Reward,Std Seq Reward,Mean Originality Score,P value,T stat,Significance
0,random_runs_dsac_seeds,0.914,0.037,0.9000,1.000000,0.000000,ns
1,random_runs_dsac_seeds_min,0.921,0.042,0.9991,0.679390,-0.420183,ns
2,random_runs_dsac_seeds_var,0.951,0.017,0.9989,0.013935,-2.848331,*
3,random_runs_dsac_seeds_bootstrapping,0.932,0.026,0.9989,0.226337,-1.257772,ns
4,random_runs_dsac_seeds_div,0.952,0.011,0.9985,0.009908,-3.140284,**
5,random_runs_dsac_seeds_prior,0.939,0.023,0.9997,0.090726,-1.807040,ns
6,random_runs_dsac_seeds_small,0.894,0.046,0.9989,0.303283,1.061183,ns
